In [16]:
import re
import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files

In [17]:
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC).xlsx to Blank Quiz (Responses 3 TIC) (2).xlsx
File terbaca: Blank Quiz (Responses 3 TIC) (2).xlsx


In [18]:
# Baca semua sheet jadi dict {nama_sheet: dataframe}
sheets = pd.read_excel(nama_file, sheet_name=None)
print("Sheet terbaca:", list(sheets.keys()))

# Butir = kolom yang judulnya berformat 'Dimensi [pernyataan]' (punya tanda '[')
# Cara ini otomatis membuang kolom demografi, berapa pun jumlahnya tiap sheet.
frames = []
for nama_sheet, d in sheets.items():
    item_cols = [c for c in d.columns if "[" in str(c)]
    frames.append(d[item_cols])
butir = pd.concat(frames, axis=1)

print("Jumlah responden :", len(butir))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

Sheet terbaca: ['Form Responses 1']
Jumlah responden : 40
Jumlah butir     : 29
Data kosong      : 0


In [19]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Sangat Tidak Setuju (STJ)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")
print("Sel gagal dikonversi:", int(num.isna().sum().sum()))  # harus 0
num.head()

Sel gagal dikonversi: 0


/tmp/ipykernel_1701/524734276.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Respon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Respon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Respon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Respon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Respon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5,5,4,4,4,4,4,5,4,4,...,2,5,4,5,4,5,5,4,4,5
1,4,4,4,4,4,4,4,5,4,4,...,4,2,4,4,4,5,4,5,5,5
2,5,5,4,4,4,4,4,4,4,4,...,2,4,5,4,2,5,5,5,4,5
3,5,4,5,2,4,5,5,4,4,2,...,2,4,4,4,4,5,4,4,4,4
4,5,5,4,4,5,5,5,4,2,4,...,2,2,2,2,4,5,5,4,5,5


In [20]:
dimensi = {}
for kolom in butir.columns:
    # ambil teks sebelum '[' sebagai nama dimensi, lalu rapikan spasi/typo
    nama = re.match(r"^(.*?)\s*\[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Respon emosional: 5 butir


In [21]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

n = 40 | r tabel (α=0,05; df=38) = 0.312


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.419,0.222,0.0071,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.390,0.224,0.0129,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.626,0.418,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.506,0.205,0.0009,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.793,0.597,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.634,0.424,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.231,-0.061,0.1510,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.578,0.437,0.0001,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.708,0.541,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.804,0.651,0.0000,Valid


In [22]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                               "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({"Dimensi": nama, "Butir": f"B{i}",
                                  "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik.,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.531
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.533
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.456
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.547
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.341
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.453
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.550


In [23]:
# ====== BUANG BUTIR TIDAK VALID, LALU UJI ULANG ======
# Format: {kata kunci nama dimensi: [nomor butir yang dibuang]}
butir_dibuang = {
    "Penggunaan AI": [7],       # B7 Penggunaan AI
    "Aspek Psikologis": [7],    # B7 Aspek Psikologis
}

# Bangun ulang pengelompokan dimensi tanpa butir yang dibuang
dimensi2, dibuang = {}, []
for nama, daftar in dimensi.items():
    buang_no = next((v for k, v in butir_dibuang.items() if k.lower() in nama.lower()), [])
    dimensi2[nama] = [kol for i, kol in enumerate(daftar, 1) if i not in buang_no]
    dibuang += [f"{nama} B{i}" for i, kol in enumerate(daftar, 1) if i in buang_no]

print("Butir dibuang:", dibuang, "\n")

# r tabel (jumlah responden tidak berubah)
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel = {r_tabel:.3f}\n")

def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

# Validitas ulang
hasil_validitas2 = []
for nama, daftar in dimensi2.items():
    x = num[daftar]; total = x.sum(axis=1)
    for i, kol in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kol], total)
        rk, _ = stats.pearsonr(x[kol], total - x[kol])
        hasil_validitas2.append({"Dimensi": nama, "Butir": f"B{i}",
                                 "r hitung": round(r, 3), "Corrected r": round(rk, 3),
                                 "Keterangan": "Valid" if r > r_tabel else "Tidak valid"})
tabel_validitas2 = pd.DataFrame(hasil_validitas2)

# Reliabilitas ulang
hasil_reliabilitas2 = []
for nama, daftar in dimensi2.items():
    a = cronbach_alpha(num[daftar])
    hasil_reliabilitas2.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                                "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
tabel_reliabilitas2 = pd.DataFrame(hasil_reliabilitas2)

display(tabel_validitas2)
display(tabel_reliabilitas2)

Butir dibuang: ['Penggunaan AI pada tugas atau projek kuliah B7', 'Aspek Psikologis penggunaan AI pada tugas atau projek kuliah B7'] 

n = 40 | r tabel = 0.312



,Dimensi,Butir,r hitung,Corrected r,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.447,0.249,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.474,0.314,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.612,0.393,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.578,0.287,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.811,0.621,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.616,0.392,Valid
6,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.607,0.477,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.739,0.591,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.790,0.637,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B4,0.657,0.464,Valid


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,6,0.639,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,6,0.769,Reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik.,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel


In [25]:
with pd.ExcelWriter("hasil_uji_setelah_buang.xlsx") as w:
    tabel_validitas2.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas2.to_excel(w, sheet_name="Reliabilitas", index=False)
files.download("hasil_uji_setelah_buang.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>